# Explore here

In [ ]:
import os
import shutil

# 1. Donde están las fotos 
origen = r'C:\Users\PC\Downloads\dogs-vs-cats\dogs-vs-cats\train'

# 2. Donde quiero que esten para el proyecto (tu carpeta de trabajo)
destino = r'C:\Users\PC\Desktop\4geeks Ejercicios\Clasficador_Imagenes\data\train'

# Crear las carpetas de destino si no existen
os.makedirs(os.path.join(destino, 'dogs'), exist_ok=True)
os.makedirs(os.path.join(destino, 'cats'), exist_ok=True)

# 3. Mover las fotos
print("Organizando archivos, espera un momento...")
for archivo in os.listdir(origen):
    if archivo.startswith("dog"):
        shutil.move(os.path.join(origen, archivo), os.path.join(destino, 'dogs', archivo))
    elif archivo.startswith("cat"):
        shutil.move(os.path.join(origen, archivo), os.path.join(destino, 'cats', archivo))

print("¡Listo! Tus imágenes ya están en la estructura correcta.")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Configuración del generador 
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# 2. Carga de datos apuntando a la carpeta 'data/train' 
train_gen = datagen.flow_from_directory(
    r'C:\Users\PC\Desktop\4geeks Ejercicios\Clasficador_Imagenes\data\train', 
    target_size=(200, 200),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    r'C:\Users\PC\Desktop\4geeks Ejercicios\Clasficador_Imagenes\data\train',
    target_size=(200, 200),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
import matplotlib.pyplot as plt

# se extrae un lote de prueba del generador
imagenes, etiquetas = next(train_gen)

#  se muestra una cuadrícula de 3x3 con las primeras 9 imágenes
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagenes[i])
    # Keras asigna las clases en orden alfabético: cats=0, dogs=1
    clase = "Perro" if etiquetas[i][1] == 1 else "Gato"
    plt.title(f"Clase: {clase}")
    plt.axis("off")
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense

model = Sequential([
    # Bloque 1
    Conv2D(input_shape=(200, 200, 3), filters=64, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=64, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPool2D(pool_size=(2,2), strides=(2,2)),
    
    # Bloque 2
    Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPool2D(pool_size=(2,2), strides=(2,2)),
    
    # Bloque 3
    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPool2D(pool_size=(2,2), strides=(2,2)),
    
    # Bloque 4
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPool2D(pool_size=(2,2), strides=(2,2)),
    
    # Bloque 5
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPool2D(pool_size=(2,2), strides=(2,2)),
    
    # Capas Densas (Clasificador final)
    Flatten(),
    Dense(units=4096, activation="relu"),
    Dense(units=4096, activation="relu"),
    Dense(units=2, activation="softmax")
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Paso 4: Callbacks de optimización
checkpoint = ModelCheckpoint("mejor_modelo.keras", monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

# Entrenamiento del modelo
print("Iniciando entrenamiento... (esto puede tardar dependiendo de tu hardware)")
historial = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10, 
    callbacks=[checkpoint, early_stop]
)

# Paso 5: Guardado final en disco
model.save('modelo_final_vgg16.keras')
print("¡Entrenamiento completado y modelo guardado con éxito!")